# Causality dataset creation

Using the output from models 8 and 9 to create 4 new datasets, merged with climate data

Model 8:
1. Synapsida
2. Reptilia All

Model 9:
1. Synapsida
2. Reptilia Terres


Dataset format:
1. Genera name: one row per species
2. Lat range value for that species
3. Biome presence binary variables
4. ts: from combined combined_10_se_est.txts
5. te: from combined combined_10_se_est.txts
6. Age range: ts-te
7. Speciation rate: lam values from combined_10_per_species_rates.log. Remove 10% first iterations, calculate average per species
8. Extinction rate: mu values from combined_10_per_species_rates.log. Remove 10% first iterations, calculate average per species
9. Each climate variable (model 8 = isotopic, model 9 = BRIDGE), but per species by finding the climate variable's average value for that species' lifespan

In [1]:
import pandas as pd

## 1-3: Genera Name, Lat Range, Biome Variables

In [3]:
syn = pd.read_csv("C:/Users/SimoesLabAdmin/Documents/pt_diversity_rates/updated_occurrence_analyses/data/Perm-Trias/bdnn_trait_files/pt_synapsida_lats_file_final.csv")
syn.head()

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic
0,Abajudon,-0.297425,0,0,1,0
1,Abdalodon,-0.297425,0,0,1,0
2,Acratophorus,-0.297425,0,0,1,0
3,Adelobasileus,-0.297425,0,1,0,0
4,Aelurognathus,1.840695,0,0,1,0


# Model 8

## 4-6: TS and TE from combined_10_se_est.txts

In [8]:
syn_ts_te = pd.read_csv("C:/Users/SimoesLabAdmin/Documents/pt_diversity_rates/updated_occurrence_analyses/model_8/synapsida/combined_10_se_est.txt", sep="\t")

syn_ts_te.head

<bound method NDFrame.head of      clade           species         ts         te
0      0.0          Abajudon  91.194269  84.871176
1      0.0         Abdalodon  82.029067  81.833773
2      0.0      Acratophorus  58.355524  53.597823
3      0.0     Adelobasileus  41.763856  41.113424
4      0.0     Aelurognathus  85.112300  77.034343
..     ...               ...        ...        ...
456    0.0         Woutersia  32.677030  28.289166
457    0.0        Woznikella  59.630453  54.044935
458    0.0  Xiyukannemeyeria  71.321191  62.080532
459    0.0       Yikezhaogia  70.415737  70.149389
460    0.0      Zambiasaurus  69.903173  69.701603

[461 rows x 4 columns]>

In [ ]:
# check if two columns are exactly the same the whole way through, only proceed if true
syn['genus'].equals(syn_ts_te['species'])

True

In [ ]:
# mege syn_ts_te 'ts' and 'te' columns based on matching 'genus' and 'species' columns
syn_merged = pd.merge(syn, syn_ts_te[['species', 'ts', 'te']], left_on='genus', right_on='species', how='left')

# Check that no rows were lost. only proceed if true
syn_merged.shape[0] == syn.shape[0] == syn_ts_te.shape[0]

True

In [17]:
syn_merged['time_span'] = syn_merged['ts'] - syn_merged['te']
syn_merged.drop(columns=['species'], inplace=True)
syn_merged.head()

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span
0,Abajudon,-0.297425,0,0,1,0,91.194269,84.871176,6.323094
1,Abdalodon,-0.297425,0,0,1,0,82.029067,81.833773,0.195294
2,Acratophorus,-0.297425,0,0,1,0,58.355524,53.597823,4.757701
3,Adelobasileus,-0.297425,0,1,0,0,41.763856,41.113424,0.650432
4,Aelurognathus,1.840695,0,0,1,0,85.112300,77.034343,8.077957


In [112]:
# ts and te need + 175 added to them, because the BDNN runs had 175 subtracted from the time

syn_merged['ts'] = syn_merged['ts'] + 175
syn_merged['te'] = syn_merged['te'] + 175

syn_merged

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,avg_lambda,avg_mu
0,Abajudon,-0.297425,0,0,1,0,266.194269,259.871176,6.323094,1.482582,1.728257
1,Abdalodon,-0.297425,0,0,1,0,257.029067,256.833773,0.195294,1.096509,1.086128
2,Acratophorus,-0.297425,0,0,1,0,233.355524,228.597823,4.757701,0.469096,0.629002
3,Adelobasileus,-0.297425,0,1,0,0,216.763856,216.113424,0.650432,1.054359,1.021459
4,Aelurognathus,1.840695,0,0,1,0,260.112300,252.034343,8.077957,0.950114,0.964767
...,...,...,...,...,...,...,...,...,...,...,...
456,Woutersia,-0.297425,1,0,0,0,207.677030,203.289166,4.387864,1.652917,1.330131
457,Woznikella,-0.064908,1,0,0,0,234.630453,229.044935,5.585518,0.401471,0.765670
458,Xiyukannemeyeria,-0.255124,1,0,0,0,246.321191,237.080532,9.240660,3.426712,1.410922
459,Yikezhaogia,-0.297425,1,0,0,0,245.415737,245.149389,0.266349,1.989161,0.458612


## 7-8: Speciation and Extinction Rates from combined_10_per_species_rates.log
Remove 10% burn-in

In [40]:
syn_lam_mu = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_8\\synapsida\\combined_10_per_species_rates.log", sep="\t")
syn_lam_mu.head()

,iteration,Abajudon_lam,Abdalodon_lam,Acratophorus_lam,Adelobasileus_lam,Aelurognathus_lam,Aelurosaurus_lam,Aelurosuchus_lam,Agudotherium_lam,Akidnognathus_lam,...,Vivaxosaurus_mu,Wadiasaurus_mu,Walteria_mu,Watongia_mu,Woutersia_mu,Woznikella_mu,Xiyukannemeyeria_mu,Yikezhaogia_mu,Zambiasaurus_mu,Unnamed: 923
0,0,1.432363,0.192334,2.495371,0.215373,1.061239,0.902277,0.316560,0.153682,0.033715,...,1.125017,0.995606,0.192807,1.970922,1.696017,0.068799,0.270910,1.083543,0.434677,NaN
1,1,1.311978,0.415080,1.297661,0.935071,0.036869,1.768891,0.484195,0.142536,1.752604,...,0.229763,2.145557,0.614548,1.970431,0.586383,1.011501,0.555901,0.559553,1.639268,NaN
2,2,0.718786,0.207362,1.009345,0.748365,0.963845,1.299296,0.517905,0.358344,1.299296,...,1.153353,1.659447,0.398429,1.952805,2.080261,0.071758,0.609455,1.472993,0.632896,NaN
3,3,0.576240,0.244166,1.128834,0.171877,0.567626,0.146110,0.311134,0.221291,1.399652,...,0.708521,0.804911,0.300521,3.739956,0.817943,1.155537,0.807538,2.106496,0.568857,NaN
4,4,0.236951,0.523158,1.592728,0.298961,0.164323,0.238942,0.497991,0.063178,0.315313,...,0.300034,0.467697,0.293405,2.669616,0.966792,1.089364,0.396220,0.505278,0.894914,NaN


Not sure what that last column is. Checking to see if all the other combined_10_per_species_rates.log files have an "Unnamed" column last

In [42]:
syn_lam_mu.columns

Index(['iteration', 'Abajudon_lam', 'Abdalodon_lam', 'Acratophorus_lam',
       'Adelobasileus_lam', 'Aelurognathus_lam', 'Aelurosaurus_lam',
       'Aelurosuchus_lam', 'Agudotherium_lam', 'Akidnognathus_lam',
       ...
       'Vivaxosaurus_mu', 'Wadiasaurus_mu', 'Walteria_mu', 'Watongia_mu',
       'Woutersia_mu', 'Woznikella_mu', 'Xiyukannemeyeria_mu',
       'Yikezhaogia_mu', 'Zambiasaurus_mu', 'Unnamed: 923'],
      dtype='object', length=924)

In [43]:
rep_all_lam_mu = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_8\\reptilia_all_offset_biohpc\\combined_10_per_species_rates.log", sep="\t")
rep_all_lam_mu.head()

,iteration,Abyssomedon_lam,Acadiella_lam,Acaenasuchus_lam,Acallosuchus_lam,Acerosodontosaurus_lam,Acleistorhinus_lam,Acrodenta_lam,Actiosaurus_lam,Adamanasuchus_lam,...,Yimenosaurus_mu,Youngetta_mu,Youngina_mu,Youngosuchus_mu,Yunguisaurus_mu,Yunnanosaurus_mu,Zanclodon_mu,Zhongjiania_mu,Zupaysaurus_mu,Unnamed: 1365
0,0,0.191855,0.081314,0.528232,0.371912,0.222653,1.164836,5.142081,1.630347,0.405571,...,2.356997,4.661170,1.209675,2.118629,2.592834,2.356997,0.752621,2.761873,0.461316,NaN
1,1,0.085859,0.323477,0.584338,0.260943,0.191425,2.653140,2.462295,1.533863,0.490450,...,2.051999,5.032406,0.880166,1.770561,3.518379,2.051999,0.689401,2.532971,0.012752,NaN
2,2,0.219212,0.786963,0.992313,0.160403,0.191799,0.907357,0.260420,1.016090,0.424271,...,2.406836,6.131269,1.151289,1.497260,2.470497,2.406836,0.760112,3.431509,0.013362,NaN
3,3,0.142135,0.265387,1.780442,0.167748,0.092720,0.435629,5.957392,1.746553,0.580494,...,3.298544,8.233369,0.543015,2.075778,2.348099,3.298544,0.651361,1.813118,0.189167,NaN
4,4,0.202969,0.239592,1.479798,0.097375,0.084421,0.292821,2.618581,1.311754,0.354102,...,1.711068,3.622490,0.287200,2.349461,1.446743,1.711068,0.398752,2.492516,0.109786,NaN


In [47]:
model_9_rep_terr_lam_mu = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_9\\reptilia_terr_new_dates\\combined_10_per_species_rates.log", sep="\t")
model_9_rep_terr_lam_mu.head()

,iteration,Abyssomedon_lam,Acadiella_lam,Acaenasuchus_lam,Acallosuchus_lam,Acerosodontosaurus_lam,Acleistorhinus_lam,Acrodenta_lam,Actiosaurus_lam,Adamanasuchus_lam,...,Yarasuchus_mu,Yimenosaurus_mu,Youngetta_mu,Youngina_mu,Youngosuchus_mu,Yunnanosaurus_mu,Zanclodon_mu,Zhongjiania_mu,Zupaysaurus_mu,Unnamed: 1111
0,0,0.142558,0.360045,1.087176,0.200497,1.611366,1.142949,4.244504,0.520488,0.199108,...,0.243140,2.099861,1.598850,1.242583,2.140484,2.099861,1.332450,1.581081,0.172798,NaN
1,1,0.278041,0.834313,1.907945,0.268821,1.302542,0.095144,3.598002,0.869982,0.457500,...,0.195666,1.309015,0.764832,0.082108,2.582038,1.309015,0.386221,1.923359,0.305835,NaN
2,2,0.213015,3.809120,1.214255,0.244011,3.234365,0.794410,2.934200,0.569384,0.440263,...,0.155695,1.016120,0.754411,0.556010,2.199347,1.016120,2.101639,2.261261,0.277221,NaN
3,3,0.126651,0.414047,0.964006,0.186965,1.130012,0.204423,2.748212,0.837615,0.530794,...,1.400752,1.381183,0.927234,1.985258,1.496028,1.381183,0.500985,1.255332,0.155841,NaN
4,4,0.328657,0.389101,2.287044,0.245272,3.144704,0.397394,2.442949,0.776871,0.383108,...,1.506006,1.396184,0.435542,1.094605,1.988060,1.396184,1.352775,1.299860,0.227769,NaN


In [48]:
model_9_syn_lam_mu = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_9\\synapsida_new_dates\\combined_10_per_species_rates.log", sep="\t")
model_9_syn_lam_mu.head()

,iteration,Abajudon_lam,Abdalodon_lam,Acratophorus_lam,Adelobasileus_lam,Aelurognathus_lam,Aelurosaurus_lam,Aelurosuchus_lam,Agudotherium_lam,Akidnognathus_lam,...,Vivaxosaurus_mu,Wadiasaurus_mu,Walteria_mu,Watongia_mu,Woutersia_mu,Woznikella_mu,Xiyukannemeyeria_mu,Yikezhaogia_mu,Zambiasaurus_mu,Unnamed: 923
0,0,6.183834,0.769476,0.160111,1.109060,0.484236,0.267567,2.732655,0.195528,0.267567,...,0.688952,1.602710,0.363748,4.037329,0.960109,0.197472,2.057631,0.471607,1.602710,NaN
1,1,0.908327,0.984794,0.312092,0.322363,1.028042,0.509901,1.775670,0.166224,0.071693,...,1.123402,0.811241,0.360284,3.792530,0.505776,0.040056,0.250681,0.184454,0.811241,NaN
2,2,0.709842,0.699869,0.210189,0.909070,1.912337,0.111100,1.277160,0.210189,0.111100,...,0.550622,1.081537,0.371697,3.480774,0.516661,0.384072,0.464022,0.510882,1.081537,NaN
3,3,0.581824,0.674504,0.177359,0.310284,0.840013,0.229051,1.489172,0.390995,0.229051,...,0.630454,0.965039,0.305003,2.211575,0.656363,0.429314,0.459359,0.825048,0.965039,NaN
4,4,5.723053,0.776792,0.205362,0.810451,0.526058,1.975051,0.949561,0.177889,0.170092,...,0.498447,0.836465,0.286584,6.403345,1.038114,0.820154,0.993928,0.624640,0.836465,NaN


In [49]:
# Thankfully they all have that column last, so we can drop it from all dataframes by index

syn_lam_mu_dropped = syn_lam_mu.drop(columns=syn_lam_mu.columns[-1])
syn_lam_mu_dropped.head()   

,iteration,Abajudon_lam,Abdalodon_lam,Acratophorus_lam,Adelobasileus_lam,Aelurognathus_lam,Aelurosaurus_lam,Aelurosuchus_lam,Agudotherium_lam,Akidnognathus_lam,...,Vinceria_mu,Vivaxosaurus_mu,Wadiasaurus_mu,Walteria_mu,Watongia_mu,Woutersia_mu,Woznikella_mu,Xiyukannemeyeria_mu,Yikezhaogia_mu,Zambiasaurus_mu
0,0,1.432363,0.192334,2.495371,0.215373,1.061239,0.902277,0.316560,0.153682,0.033715,...,0.434677,1.125017,0.995606,0.192807,1.970922,1.696017,0.068799,0.270910,1.083543,0.434677
1,1,1.311978,0.415080,1.297661,0.935071,0.036869,1.768891,0.484195,0.142536,1.752604,...,1.639268,0.229763,2.145557,0.614548,1.970431,0.586383,1.011501,0.555901,0.559553,1.639268
2,2,0.718786,0.207362,1.009345,0.748365,0.963845,1.299296,0.517905,0.358344,1.299296,...,0.632896,1.153353,1.659447,0.398429,1.952805,2.080261,0.071758,0.609455,1.472993,0.632896
3,3,0.576240,0.244166,1.128834,0.171877,0.567626,0.146110,0.311134,0.221291,1.399652,...,0.894895,0.708521,0.804911,0.300521,3.739956,0.817943,1.155537,0.807538,2.106496,0.568857
4,4,0.236951,0.523158,1.592728,0.298961,0.164323,0.238942,0.497991,0.063178,0.315313,...,0.894914,0.300034,0.467697,0.293405,2.669616,0.966792,1.089364,0.396220,0.505278,0.894914


In [51]:
# making sure there's only a one column difference after dropping
syn_lam_mu.shape, syn_lam_mu_dropped.shape

((1000, 924), (1000, 923))

In [ ]:
# drop rows 1-100 inclusive
syn_lam_mu_dropped = syn_lam_mu_dropped.drop(index=range(0, 100))
syn_lam_mu_dropped.shape[0] == 900 # all combined logs were resampled to 100, so they should all have 1000 rows originally, 900 post dropping burn in

(900, 923)

In [53]:
# Get a new dataset which = the average value of each column, maintaining the column heads
syn_lam_mu_dropped_avg = pd.DataFrame(syn_lam_mu_dropped.mean()).T
syn_lam_mu_dropped_avg.shape[1] == syn_lam_mu_dropped.shape[1]

True

In [65]:
syn_lam_mu_dropped_avg.head()

,iteration,Abajudon_lam,Abdalodon_lam,Acratophorus_lam,Adelobasileus_lam,Aelurognathus_lam,Aelurosaurus_lam,Aelurosuchus_lam,Agudotherium_lam,Akidnognathus_lam,...,Vinceria_mu,Vivaxosaurus_mu,Wadiasaurus_mu,Walteria_mu,Watongia_mu,Woutersia_mu,Woznikella_mu,Xiyukannemeyeria_mu,Yikezhaogia_mu,Zambiasaurus_mu
0,549.5,1.482582,1.096509,0.469096,1.054359,0.950114,0.890719,1.492644,0.388194,0.624368,...,0.942098,0.745403,0.820456,0.997598,1.2897,1.330131,0.76567,1.410922,0.458612,0.974695


In [55]:
# Separate lambda and mu columns into their own dataframes
syn_lambda = syn_lam_mu_dropped_avg.filter(like='_lam')
syn_mu = syn_lam_mu_dropped_avg.filter(like='_mu')
syn_lambda.head()

,Abajudon_lam,Abdalodon_lam,Acratophorus_lam,Adelobasileus_lam,Aelurognathus_lam,Aelurosaurus_lam,Aelurosuchus_lam,Agudotherium_lam,Akidnognathus_lam,Aleodon_lam,...,Vinceria_lam,Vivaxosaurus_lam,Wadiasaurus_lam,Walteria_lam,Watongia_lam,Woutersia_lam,Woznikella_lam,Xiyukannemeyeria_lam,Yikezhaogia_lam,Zambiasaurus_lam
0,1.482582,1.096509,0.469096,1.054359,0.950114,0.890719,1.492644,0.388194,0.624368,1.142229,...,1.704392,0.282749,1.014446,0.582476,0.992165,1.652917,0.401471,3.426712,1.989161,1.356252


In [68]:
syn_mu.head()

species,Abajudon_mu,Abdalodon_mu,Acratophorus_mu,Adelobasileus_mu,Aelurognathus_mu,Aelurosaurus_mu,Aelurosuchus_mu,Agudotherium_mu,Akidnognathus_mu,Aleodon_mu,...,Vinceria_mu,Vivaxosaurus_mu,Wadiasaurus_mu,Walteria_mu,Watongia_mu,Woutersia_mu,Woznikella_mu,Xiyukannemeyeria_mu,Yikezhaogia_mu,Zambiasaurus_mu
0,1.728257,1.086128,0.629002,1.021459,0.964767,0.527475,1.128484,0.455886,0.474949,0.22833,...,0.942098,0.745403,0.820456,0.997598,1.2897,1.330131,0.76567,1.410922,0.458612,0.974695


In [ ]:
# making sure we didn't lose any columns. The +1 is for the iterations column
syn_lambda.shape[1] + syn_mu.shape[1] + 1 == syn_lam_mu_dropped_avg.shape[1]

True

In [57]:
# syn_lambda should = syn_mu in number of columns
syn_lambda.shape[1] == syn_mu.shape[1]

True

In [96]:
# Transpose both dataframes to have species as rows and averages as columns. 
syn_lambda_t = syn_lambda.T.reset_index()
syn_mu_t = syn_mu.T.reset_index()
syn_lambda_t.head(), syn_mu_t.head()

(             species         0
 0       Abajudon_lam  1.482582
 1      Abdalodon_lam  1.096509
 2   Acratophorus_lam  0.469096
 3  Adelobasileus_lam  1.054359
 4  Aelurognathus_lam  0.950114,
             species         0
 0       Abajudon_mu  1.728257
 1      Abdalodon_mu  1.086128
 2   Acratophorus_mu  0.629002
 3  Adelobasileus_mu  1.021459
 4  Aelurognathus_mu  0.964767)

In [97]:
syn_lambda_t.columns

Index(['species', 0], dtype='object')

In [102]:
syn_lambda_t.rename(columns={0: 'avg_lambda'}, inplace=True)
syn_mu_t.rename(columns={0: 'avg_mu'}, inplace=True)
syn_lambda_t.head(), syn_mu_t.head()

(             species  avg_lambda
 0       Abajudon_lam    1.482582
 1      Abdalodon_lam    1.096509
 2   Acratophorus_lam    0.469096
 3  Adelobasileus_lam    1.054359
 4  Aelurognathus_lam    0.950114,
             species    avg_mu
 0       Abajudon_mu  1.728257
 1      Abdalodon_mu  1.086128
 2   Acratophorus_mu  0.629002
 3  Adelobasileus_mu  1.021459
 4  Aelurognathus_mu  0.964767)

In [103]:
syn_lambda_t.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 461 entries, 0 to 460
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   species     461 non-null    object 
 1   avg_lambda  461 non-null    float64
dtypes: float64(1), object(1)
memory usage: 7.3+ KB


In [104]:
# strip the suffixes from all the entries in the 'species' column of syn_lambda_t
syn_lambda_t_stripped = syn_lambda_t.copy()
syn_mu_t_stripped = syn_mu_t.copy()
syn_lambda_t_stripped['species'] = syn_lambda_t_stripped['species'].str.replace('_lam', '')
syn_mu_t_stripped['species'] = syn_mu_t_stripped['species'].str.replace('_mu', '')

In [105]:
syn_lambda_t_stripped, syn_mu_t_stripped

(              species  avg_lambda
 0            Abajudon    1.482582
 1           Abdalodon    1.096509
 2        Acratophorus    0.469096
 3       Adelobasileus    1.054359
 4       Aelurognathus    0.950114
 ..                ...         ...
 456         Woutersia    1.652917
 457        Woznikella    0.401471
 458  Xiyukannemeyeria    3.426712
 459       Yikezhaogia    1.989161
 460      Zambiasaurus    1.356252
 
 [461 rows x 2 columns],
               species    avg_mu
 0            Abajudon  1.728257
 1           Abdalodon  1.086128
 2        Acratophorus  0.629002
 3       Adelobasileus  1.021459
 4       Aelurognathus  0.964767
 ..                ...       ...
 456         Woutersia  1.330131
 457        Woznikella  0.765670
 458  Xiyukannemeyeria  1.410922
 459       Yikezhaogia  0.458612
 460      Zambiasaurus  0.974695
 
 [461 rows x 2 columns])

In [106]:
syn_lambda_t_stripped.shape[0] == syn_mu_t_stripped.shape[0] == syn_merged.shape[0]

True

In [107]:
syn_merged = pd.merge(syn_merged, syn_lambda_t_stripped, left_on='genus', right_on='species', how='left')
syn_merged = pd.merge(syn_merged, syn_mu_t_stripped, left_on='genus', right_on='species', how='left')

In [108]:
syn_merged

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,species_x,avg_lambda,species_y,avg_mu
0,Abajudon,-0.297425,0,0,1,0,91.194269,84.871176,6.323094,Abajudon,1.482582,Abajudon,1.728257
1,Abdalodon,-0.297425,0,0,1,0,82.029067,81.833773,0.195294,Abdalodon,1.096509,Abdalodon,1.086128
2,Acratophorus,-0.297425,0,0,1,0,58.355524,53.597823,4.757701,Acratophorus,0.469096,Acratophorus,0.629002
3,Adelobasileus,-0.297425,0,1,0,0,41.763856,41.113424,0.650432,Adelobasileus,1.054359,Adelobasileus,1.021459
4,Aelurognathus,1.840695,0,0,1,0,85.112300,77.034343,8.077957,Aelurognathus,0.950114,Aelurognathus,0.964767
...,...,...,...,...,...,...,...,...,...,...,...,...,...
456,Woutersia,-0.297425,1,0,0,0,32.677030,28.289166,4.387864,Woutersia,1.652917,Woutersia,1.330131
457,Woznikella,-0.064908,1,0,0,0,59.630453,54.044935,5.585518,Woznikella,0.401471,Woznikella,0.765670
458,Xiyukannemeyeria,-0.255124,1,0,0,0,71.321191,62.080532,9.240660,Xiyukannemeyeria,3.426712,Xiyukannemeyeria,1.410922
459,Yikezhaogia,-0.297425,1,0,0,0,70.415737,70.149389,0.266349,Yikezhaogia,1.989161,Yikezhaogia,0.458612


In [109]:
syn_merged.drop(columns=['species_x', 'species_y'], inplace=True)

In [113]:
syn_merged

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,avg_lambda,avg_mu
0,Abajudon,-0.297425,0,0,1,0,266.194269,259.871176,6.323094,1.482582,1.728257
1,Abdalodon,-0.297425,0,0,1,0,257.029067,256.833773,0.195294,1.096509,1.086128
2,Acratophorus,-0.297425,0,0,1,0,233.355524,228.597823,4.757701,0.469096,0.629002
3,Adelobasileus,-0.297425,0,1,0,0,216.763856,216.113424,0.650432,1.054359,1.021459
4,Aelurognathus,1.840695,0,0,1,0,260.112300,252.034343,8.077957,0.950114,0.964767
...,...,...,...,...,...,...,...,...,...,...,...
456,Woutersia,-0.297425,1,0,0,0,207.677030,203.289166,4.387864,1.652917,1.330131
457,Woznikella,-0.064908,1,0,0,0,234.630453,229.044935,5.585518,0.401471,0.765670
458,Xiyukannemeyeria,-0.255124,1,0,0,0,246.321191,237.080532,9.240660,3.426712,1.410922
459,Yikezhaogia,-0.297425,1,0,0,0,245.415737,245.149389,0.266349,1.989161,0.458612


## 9. Isotopic Data

Each genus existed from a certain time of speciation (ts) to time of extinction (te). We want to assign that full time span one value per environmental variable, representing the average environmental conditions during that genus's life span. I have climate data across time. I need to:

In [ ]:
# Some time spans are extremely short, <1, which means they'll have trouble being assigned a climate variable. I'll have to handle these edge cases in my function
syn_merged['time_span'].describe()

count    461.000000
mean       4.643856
std        5.751556
min        0.074908
25%        0.431390
50%        2.800586
75%        7.019690
max       37.630210
Name: time_span, dtype: float64

In [ ]:
# syn_merged_subset = syn_merged[['genus', 'ts', 'te']]
# syn_merged_subset.to_csv("C:/Users/SimoesLabAdmin/Documents/pt_diversity_rates/syn_merged_subset.txt", sep="\t", index=False)

In [117]:
isotopic = pd.read_csv("C:/Users/SimoesLabAdmin/Documents/pt_diversity_rates/updated_occurrence_analyses/data/Perm-Trias/songEA_isotopic_data_pt_1myr/isotopic_1myr_filtered_final.txt", sep="\t")
isotopic.head()

,Time,mean_pt_1myr_z_trans,Mod_R_deltaTMyr_pt_1myr_z_trans
0,300,0.135311,-0.865252
1,299,-0.192132,-0.486496
2,298,-0.939830,0.444172
3,297,-0.595793,-0.449748
4,296,-0.107576,-0.130457


### Average and Nearest Neighbors Method

In [ ]:
# Isotopic has "Time" and syn_merged_nn has "ts" and "te". I want to get the average value of each column in Isotopic for the lifespan of each genus in syn_merged_nn, then add those averages as new columns in syn_merged_nn

syn_merged_nn = syn_merged.copy()
syn_merged_nn['mean_pt_1myr_z_trans'] = None
syn_merged_nn['Mod_R_deltaTMyr_pt_1myr_z_trans'] = None

for index, row in syn_merged_nn.iterrows():
    ts = row['ts']
    te = row['te']
    genus = row['genus']
    
    # Filter isotopic data for the time span of the genus
    isotopic_filtered = isotopic[(isotopic['Time'] <= ts) & (isotopic['Time'] >= te)]
    
    # Calculate mean values for the relevant columns
    mean_values = isotopic_filtered[['mean_pt_1myr_z_trans', 'Mod_R_deltaTMyr_pt_1myr_z_trans']].mean()

    # Assign mean values to the syn_merged_nn dataframe
    syn_merged_nn.at[index, 'mean_pt_1myr_z_trans'] = mean_values['mean_pt_1myr_z_trans']
    syn_merged_nn.at[index, 'Mod_R_deltaTMyr_pt_1myr_z_trans'] = mean_values['Mod_R_deltaTMyr_pt_1myr_z_trans']

    # Now, if the row has a null value for either of the new columns, we can try to assign the value from the closest time point in isotopic data
    if pd.isna(syn_merged_nn.at[index, 'mean_pt_1myr_z_trans']) or pd.isna(syn_merged_nn.at[index, 'Mod_R_deltaTMyr_pt_1myr_z_trans']):
        # Find the closest time point in isotopic data
        closest_time = isotopic.iloc[(isotopic['Time'] - ((ts + te) / 2)).abs().argsort()[:1]]
        
        # Assign the values from the closest time point
        syn_merged_nn.at[index, 'mean_pt_1myr_z_trans'] = closest_time['mean_pt_1myr_z_trans'].values[0]
        syn_merged_nn.at[index, 'Mod_R_deltaTMyr_pt_1myr_z_trans'] = closest_time['Mod_R_deltaTMyr_pt_1myr_z_trans'].values[0]

# Check that no rows were lost. only proceed if true
assert syn_merged_nn.shape[0] == syn.shape[0], "Rows were lost during merge!"

In [ ]:
ts=216.76, te=216.11

In [140]:
syn_merged_nn

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,avg_lambda,avg_mu,mean_pt_1myr_z_trans,Mod_R_deltaTMyr_pt_1myr_z_trans
0,Abajudon,-0.297425,0,0,1,0,266.194269,259.871176,6.323094,1.482582,1.728257,-0.917951,0.704704
1,Abdalodon,-0.297425,0,0,1,0,257.029067,256.833773,0.195294,1.096509,1.086128,-0.016673,-0.859216
2,Acratophorus,-0.297425,0,0,1,0,233.355524,228.597823,4.757701,0.469096,0.629002,1.005908,0.025418
3,Adelobasileus,-0.297425,0,1,0,0,216.763856,216.113424,0.650432,1.054359,1.021459,-0.16298,-0.763084
4,Aelurognathus,1.840695,0,0,1,0,260.112300,252.034343,8.077957,0.950114,0.964767,-0.544651,-0.226106
...,...,...,...,...,...,...,...,...,...,...,...,...,...
456,Woutersia,-0.297425,1,0,0,0,207.677030,203.289166,4.387864,1.652917,1.330131,0.099289,-0.628074
457,Woznikella,-0.064908,1,0,0,0,234.630453,229.044935,5.585518,0.401471,0.765670,1.025593,-0.426136
458,Xiyukannemeyeria,-0.255124,1,0,0,0,246.321191,237.080532,9.240660,3.426712,1.410922,1.304997,0.275994
459,Yikezhaogia,-0.297425,1,0,0,0,245.415737,245.149389,0.266349,1.989161,0.458612,0.814772,-0.796539


In [141]:
syn_merged_nn.isna().sum()

genus                              0
lat_range_z_trans                  0
Temperate_N                        0
Tropical                           0
Temperate_S                        0
Antarctic                          0
ts                                 0
te                                 0
time_span                          0
avg_lambda                         0
avg_mu                             0
mean_pt_1myr_z_trans               0
Mod_R_deltaTMyr_pt_1myr_z_trans    0
dtype: int64

In [ ]:
nulls = syn_merged_nn[syn_merged_nn['mean_pt_1myr_z_trans'].isna() | syn_merged_nn['Mod_R_deltaTMyr_pt_1myr_z_trans'].isna()]
nulls.describe()

,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,avg_lambda,avg_mu
count,1.330000e+02,133.000000,133.000000,133.000000,133.000000,133.000000,133.000000,133.000000,133.000000,133.000000
mean,-2.974246e-01,0.323308,0.157895,0.511278,0.007519,253.719006,253.403466,0.315540,1.459533,1.424551
std,3.900472e-16,0.469508,0.366021,0.501763,0.086711,18.854696,18.875210,0.200864,1.094565,0.878700
min,-2.974246e-01,0.000000,0.000000,0.000000,0.000000,203.818865,203.346156,0.074908,0.144761,0.194651
25%,-2.974246e-01,0.000000,0.000000,0.000000,0.000000,248.174992,248.095285,0.181323,0.887548,0.974695
50%,-2.974246e-01,0.000000,0.000000,1.000000,0.000000,256.507902,256.312923,0.208546,1.125583,1.128484
75%,-2.974246e-01,1.000000,0.000000,1.000000,0.000000,261.873385,261.165365,0.441549,1.496820,1.536277
max,-2.974246e-01,1.000000,1.000000,1.000000,1.000000,295.706109,295.191776,1.223492,5.985827,4.746130


In [127]:
nulls[nulls['time_span'] == nulls['time_span'].max()]

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,avg_lambda,avg_mu,mean_pt_1myr_z_trans,Mod_R_deltaTMyr_pt_1myr_z_trans
49,Botucaraitherium,-0.297425,0,0,1,0,218.620446,217.396954,1.223492,0.225505,0.194651,NaN,NaN


In [128]:
nulls[nulls['time_span'] >= 1]

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,avg_lambda,avg_mu,mean_pt_1myr_z_trans,Mod_R_deltaTMyr_pt_1myr_z_trans
49,Botucaraitherium,-0.297425,0,0,1,0,218.620446,217.396954,1.223492,0.225505,0.194651,NaN,NaN


In [ ]:
# Check for potential edge cases
for index, row in syn_merged_nn.iterrows():
    ts, te = row['ts'], row['te']
    overlaps = isotopic[(isotopic['Time'] <= ts) & (isotopic['Time'] >= te)]
    if len(overlaps) == 0:
        print(f"No overlap for {row['genus']}: ts={ts:.2f}, te={te:.2f}")

No overlap for Adelobasileus: ts=216.76, te=216.11
No overlap for Aelurosuchus: ts=244.68, te=244.50
No overlap for Alopecorhinus: ts=259.97, te=259.47
No overlap for Anomocephalus: ts=265.99, te=265.55
No overlap for Apsisaurus: ts=292.82, te=292.26
No overlap for Arisierpeton: ts=287.70, te=287.26
No overlap for Aulacocephalus: ts=256.94, te=256.75
No overlap for Australosyodon: ts=265.38, te=265.00
No overlap for Beishanodon: ts=248.38, te=248.30
No overlap for Biarmosuchoides: ts=265.94, te=265.57
No overlap for Blattoidealestes: ts=259.78, te=259.17
No overlap for Bolotridon: ts=244.47, te=244.25
No overlap for Botucaraitherium: ts=218.62, te=217.40
No overlap for Burnetia: ts=257.47, te=257.29
No overlap for Callibrachion: ts=287.98, te=287.61
No overlap for Candelariodon: ts=240.41, te=240.20
No overlap for Cerdorhinus: ts=256.28, te=256.09
No overlap for Cerdosuchus: ts=255.40, te=255.23
No overlap for Chalepotherium: ts=204.70, te=204.11
No overlap for Charassognathus: ts=256.

### Weighted Interpolation Method

In [ ]:

syn_merged_wi = syn_merged.copy()
# Initialize new columns in syn_merged_wi
syn_merged_wi['mean_pt_1myr_z_trans'] = None
syn_merged_wi['Mod_R_deltaTMyr_pt_1myr_z_trans'] = None

# Get sorted list of available time points for efficient lookup
available_times = sorted(isotopic['Time'].unique(), reverse=True)

# Loop through each genus
for index, row in syn_merged_wi.iterrows():
    ts = row['ts']  # Time of speciation (older)
    te = row['te']  # Time of extinction (younger)
    
    # Calculate midpoint
    midpoint = (ts + te) / 2
    
    # STEP 1: Check if any time points fall directly within the genus range
    # This handles genera that overlap with actual sampling points
    isotopic_filtered = isotopic[(isotopic['Time'] <= ts) & (isotopic['Time'] >= te)]
    
    if len(isotopic_filtered) > 0:
        # Direct overlap exists - use arithmetic mean
        mean_pt = isotopic_filtered['mean_pt_1myr_z_trans'].mean()
        mod_r = isotopic_filtered['Mod_R_deltaTMyr_pt_1myr_z_trans'].mean()
    
    else:
        # STEP 2: No direct overlap - use weighted interpolation
        
        # Find t_upper: smallest time point >= ts (closest point older than or at speciation)
        upper_candidates = [t for t in available_times if t >= ts]
        if len(upper_candidates) == 0:
            t_upper = available_times[0]  # Use oldest available if genus predates all data
        else:
            t_upper = min(upper_candidates)
        
        # Find t_lower: largest time point <= te (closest point younger than or at extinction)
        lower_candidates = [t for t in available_times if t <= te]
        if len(lower_candidates) == 0:
            t_lower = available_times[-1]  # Use youngest available if genus postdates all data
        else:
            t_lower = max(lower_candidates)
        
        # Calculate weights based on inverse distance
        # Points closer to midpoint get MORE weight
        if t_upper != t_lower:
            w_lower = (t_upper - midpoint) / (t_upper - t_lower)
            w_upper = (midpoint - t_lower) / (t_upper - t_lower)
        else:
            # Edge case: both brackets are the same point (extrapolation)
            w_lower = 1.0
            w_upper = 0.0
        
        # Get isotopic values at the bracketing points
        val_lower_pt = isotopic[isotopic['Time'] == t_lower]['mean_pt_1myr_z_trans'].values[0]
        val_upper_pt = isotopic[isotopic['Time'] == t_upper]['mean_pt_1myr_z_trans'].values[0]
        val_lower_mod = isotopic[isotopic['Time'] == t_lower]['Mod_R_deltaTMyr_pt_1myr_z_trans'].values[0]
        val_upper_mod = isotopic[isotopic['Time'] == t_upper]['Mod_R_deltaTMyr_pt_1myr_z_trans'].values[0]
        
        # Calculate weighted averages
        # value(midpoint) = value(lower) × weight(lower) + value(upper) × weight(upper)
        mean_pt = val_lower_pt * w_lower + val_upper_pt * w_upper
        mod_r = val_lower_mod * w_lower + val_upper_mod * w_upper
    
    # Assign calculated values to the dataframe
    syn_merged_wi.at[index, 'mean_pt_1myr_z_trans'] = mean_pt
    syn_merged_wi.at[index, 'Mod_R_deltaTMyr_pt_1myr_z_trans'] = mod_r

# Verify no rows were lost
assert syn_merged_wi.shape[0] == syn_merged.shape[0], "Rows were lost during merge!"

# Check for any NaN values (shouldn't happen if code is correct)
if syn_merged_wi['mean_pt_1myr_z_trans'].isna().any():
    print("Warning: Some genera have NaN values")
    print(syn_merged_wi[syn_merged_wi['mean_pt_1myr_z_trans'].isna()])
else:
    print(f"Success! All {len(syn_merged_wi)} genera have isotopic values assigned.")

# Optional: View summary statistics
print("\nSummary of assigned isotopic values:")
print(syn_merged_wi[['mean_pt_1myr_z_trans', 'Mod_R_deltaTMyr_pt_1myr_z_trans']].describe())

Success! All 461 genera have isotopic values assigned.

Summary of assigned isotopic values:
        mean_pt_1myr_z_trans  Mod_R_deltaTMyr_pt_1myr_z_trans
count             461.000000                       461.000000
unique            306.000000                       306.000000
top                -1.544949                         1.524857
freq               17.000000                        17.000000


In [138]:
syn_merged_wi

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,avg_lambda,avg_mu,mean_pt_1myr_z_trans,Mod_R_deltaTMyr_pt_1myr_z_trans
0,Abajudon,-0.297425,0,0,1,0,266.194269,259.871176,6.323094,1.482582,1.728257,-0.917951,0.704704
1,Abdalodon,-0.297425,0,0,1,0,257.029067,256.833773,0.195294,1.096509,1.086128,-0.016673,-0.859216
2,Acratophorus,-0.297425,0,0,1,0,233.355524,228.597823,4.757701,0.469096,0.629002,1.005908,0.025418
3,Adelobasileus,-0.297425,0,1,0,0,216.763856,216.113424,0.650432,1.054359,1.021459,-0.074135,-0.093139
4,Aelurognathus,1.840695,0,0,1,0,260.112300,252.034343,8.077957,0.950114,0.964767,-0.544651,-0.226106
...,...,...,...,...,...,...,...,...,...,...,...,...,...
456,Woutersia,-0.297425,1,0,0,0,207.677030,203.289166,4.387864,1.652917,1.330131,0.099289,-0.628074
457,Woznikella,-0.064908,1,0,0,0,234.630453,229.044935,5.585518,0.401471,0.765670,1.025593,-0.426136
458,Xiyukannemeyeria,-0.255124,1,0,0,0,246.321191,237.080532,9.240660,3.426712,1.410922,1.304997,0.275994
459,Yikezhaogia,-0.297425,1,0,0,0,245.415737,245.149389,0.266349,1.989161,0.458612,0.867735,-0.793066


### Debate Between the Two Methods

Both take averages in the case of genera whose life spans include multiple isotopic data points in between their ts and te! Where they differ is how they handle 

Pro of Weighted Interpolation